In [ ]:
from typing import Iterable, Callable
from itertools import chain as iterchain, combinations as itercomb
from collections import deque

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, POS9, Loc, Cell, Node, Board
from utils import count_finals, count_digits
from analytics import Locality, Target, MultiTarget
from analytics import validate, all_visible, are_visible, Node_has, neighborhood
from solving import orchestrator, solver, Resolution, Resolving, Resolver

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
async def solve_silent(initial: Board, *resolvers: Resolver):
    result = initial
    async for _, _, result in solver(initial, orchestrator(initial, *resolvers)):
        pass
    return result

In [ ]:
async def solve_logging(initial: Board, /, *resolvers: Resolver, filtout: set[str] | None = None):
    result = initial
    iterations = 0
    async for resolver, resolution, result in solver(initial, orchestrator(initial, *resolvers)):
        iterations += 1
        if filtout and resolver.__name__ in filtout:
            continue
        print(f"{iterations:03d} {resolver.__name__}", end=": ")
        if resolution.castaways:
            print("-={", " ".join(map(str, resolution.castaways)), "}", end=" ")
        if resolution.finals:
            print(":={", " ".join(map(str, resolution.finals)), "}", end=" ")
        if resolution.highlights:
            if "zone" in resolution.highlights:
                print("@", resolution.highlights["zone"], end=" ")
            print("#", end=" ")
            if "anchors" in resolution.highlights:
                print(" ".join(map(str, resolution.highlights["anchors"])), end=" ")
            if "chain" in resolution.highlights:
                print(resolution.highlights["chain"], end=" ")
        print()
    print(validate(result), "in", iterations)
    return result

## Basic

Singles in localities and their contra-neighbors


In [ ]:
def cleanup(board: Board) -> Resolving:
    """Removing drafts contradicting with neighbouring finals"""

    for zone in Locality.all():
        finalborhood = list(filter(Node.is_final, neighborhood(zone, board)))
        draftborhood = list(filter(Node.is_draft, neighborhood(zone, board)))
        for finode in finalborhood:
            dig = finode.cell.final
            assert dig is not None
            contras = list(filter(Node_has(dig), draftborhood))
            if contras:
                yield Resolution(
                    castaways=set(Target(n.loc, dig) for n in contras),
                    highlights={"anchors": {Target(finode.loc, dig)}, "zone": zone},
                )

In [ ]:
def singles(board: Board) -> Resolving:
    """Isolate singular digits in localities"""

    for zone in Locality.all():
        draftborhood = list(filter(Node.is_draft, neighborhood(zone, board)))
        for dig in DIGITS:
            family = list(filter(Node_has(dig), draftborhood))
            if len(family) == 1 and len(family[0]) > 1:
                lonesome = Target(family[0].loc, dig)
                yield Resolution(
                    finals={lonesome},
                    highlights={"zone": zone},
                )

## Multiples

Combos of N digits


In [ ]:
def all_combos(m: int):
    return map(set[int], itercomb(DIGITS, m))

#### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

Rule: remove the digits of the combo from all other cells


In [ ]:
def clean_mults(board: Board, mult: int) -> Resolving:
    """Clean out spoiled neighbours of open multiples in each zone"""
    for zone in Locality.all():
        draftborhood = set(filter(Node.is_draft, neighborhood(zone, board)))
        for combo in all_combos(mult):
            # all nodes containing only the combo
            habitat = set(filter(lambda n: n.cell <= combo, draftborhood))
            # all other neighbors containing some combo digits
            spoilers = tuple(filter(lambda n: n.cell & combo, draftborhood - habitat))

            if len(habitat) == mult and len(spoilers):
                yield Resolution(
                    castaways=set(Target(n.loc, d) for n in spoilers for d in n.cell & combo),
                    highlights={
                        "zone": zone,
                        "anchors": set(MultiTarget.new(n.loc, combo) for n in habitat),
                    },
                )


def clean_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return clean_mults(board, mult)

    resolver.__name__ = f"open_mults[{mult}]"
    return resolver

### hidden

Some n-combo contained in only n cells (within locality) // along other drafts

Rule: remove all other drafts from the cells => it becomes open


In [ ]:
def unhide_mults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Locality.all():
        draftborhood = tuple(filter(Node.is_draft, neighborhood(zone, board)))
        for combo in all_combos(mult):
            # all nodes containing some combo digits (+ some spoilers)
            habitat = tuple(filter(lambda n: n.cell & combo, draftborhood))
            # set of all actual combo digits in the nodes
            habitants = set(iterflat(n.cell & combo for n in habitat))
            # inhabited nodes with other digits
            spoiled = tuple(filter(lambda n: n.cell - combo, habitat))

            if len(habitat) == mult and len(habitants) == mult and len(spoiled):
                yield Resolution(
                    castaways=set(Target(n.loc, d) for n in spoiled for d in n.cell - combo),
                    highlights={
                        "zone": zone,
                        "anchors": set(MultiTarget.new(n.loc, combo) for n in habitat),
                    },
                )


def unhide_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return unhide_mults(board, mult)

    resolver.__name__ = f"hidden_mults[{mult}]"
    return resolver

## Links

Links represent XOR or NAND relations between drafts.

- XOR $\veebar$ corresponds to "each digit appears only once in a locality"
- NAND $\barwedge$ corrsponds to "each locality contains only different digits"

(or vise versa, I dunno)


In [ ]:
from analytics import Link, HLink, SLink

#### strong/hard links

Represent XOR relation $\veebar$

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts (of different digits) in a cell


In [ ]:
def search_hard(board: Board) -> Iterable[HLink]:
    """Search for all hard links"""

    def scan_cell(node: Node):
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield HLink((Target(node.loc, d1), Target(node.loc, d2)))

    def scan_locality(zone: Locality):
        draftborhood = tuple(filter(Node.is_draft, neighborhood(zone, board)))
        for d in DIGITS:
            family = tuple(filter(Node_has(d), draftborhood))
            if len(family) == 2:
                n1, n2 = family
                yield HLink((Target(n1.loc, d), Target(n2.loc, d)))

    for node in filter(Node.is_draft, board):
        yield from scan_cell(node)

    for zone in Locality.all():
        yield from scan_locality(zone)

#### weak/soft links

Represent NAND relation $\barwedge$

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts (of different digits) in a cell

Note: The criteria are totally independent of board content (calculating from locations only)

Visibility = soft-linkability


In [ ]:
def are_nandable(t1: Target | MultiTarget, t2: Target | MultiTarget):
    """If they can form a soft link"""
    if t1.is_singular and t2.is_singular and t1.dig == t2.dig:
        return are_visible(t1.loc, t2.loc)
    else:
        return t1.loc == t2.loc


def scan_visible(board: Board, e1: Target, e2: Target, anchors: set[Target]) -> Iterable[Target]:
    """Scan for all targets visible from the given (edges)"""
    for node in filter(Node.is_draft, board.slice(all_visible(e1.loc, e2.loc))):
        for d in node.cell:
            trg = Target(node.loc, d)
            if trg not in anchors and are_nandable(trg, e1) and are_nandable(trg, e2):
                yield trg

## Chains

Alterating link chains constituted of `~ hard ~ soft ~` and `~ soft ~ hard ~`

Lemma1: $(X \barwedge A) \cdot (A \veebar B) \cdot (B \barwedge X) \Rightarrow \neg X$

Meaning: all draft visible (soft-linkable) from some XORed points, are all invalid

Lemma2: $(X \veebar A) \cdot (A \barwedge B) \cdot (B \veebar Y) \Rightarrow (X \veebar Y)$

Meaning: a ALC (of any length) with hard edges behaves as if its edges are hard-linked


In [ ]:
from analytics import Chain

#### loop ALC

ALC with connected edges: `X ~ hard ~ ... ~ soft ~ X`

Rule: invalidate all draft visible from each soft-link in the chain


In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors = chain.anchors()

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(scan_visible(board, t1, t2, anchors))
        if len(spoilers):
            yield Resolution(
                spoilers,
                highlights={"anchors": {t1, t2}, "chain": chain},
            )

#### open ALC

ALC with hard links at its edges: `X ~ hard ~ ... ~ hard ~ Y`

Rule: invalidate all drafts visible from both edges of such chain


In [ ]:
def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    anchors = chain.anchors()
    e1, e2 = chain.edges

    spoilers = set(scan_visible(board, e1, e2, anchors))
    if len(spoilers):
        yield Resolution(
            spoilers,
            highlights={"anchors": {e1, e2}, "chain": chain},
        )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [ ]:
def expand_alc(chain: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""
    e1, e2 = chain.edges

    # closing loop
    if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and are_nandable(e1, e2):
        yield Chain.extend(chain, SLink((e2, e1)))

    anchors: set[Target] = chain.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        x1, x2 = link
        if are_nandable(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if are_nandable(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())
        if are_nandable(x2, e1):
            yield Chain.extendhead(chain, link, SLink((x2, e1)))
        if are_nandable(x1, e1):
            yield Chain.extendhead(chain, link.reversed(), SLink((x1, e1)))


def search_chains_breadth(links: Iterable[HLink], creiteria: Callable[[Chain], bool]) -> Iterable[Chain]:
    """Search for all chain matching creiteria
    breadth-first search (shortest-first)
    it yields infinitely
    breaking search is up to calling coroutine
    """
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.popleft()
        if creiteria(chain):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)


def search_chains_depth(links: Iterable[HLink], creiteria: Callable[[Chain], bool]) -> Iterable[Chain]:
    """Search for all chain matching creiteria
    depth-first search (longest-first)
    it yields infinitely
    breaking search is up to calling coroutine
    """
    frontier = deque[Chain](Chain.init(lnk) for lnk in links)  # stack
    explored = set[Chain]()
    while frontier:
        chain = frontier.pop()
        if creiteria(chain):
            yield chain
        explored.add(chain)
        frontier.extend(ext for ext in expand_alc(chain, links) if ext not in explored and ext not in frontier)

In [ ]:
def chains_(kind: str, strategy: str = "B"):
    if strategy == "D":
        searching = search_chains_depth
    elif strategy == "B":
        searching = search_chains_breadth
    else:
        raise ValueError()

    if kind == "loop":
        matching = match_loop
        resolving = resolve_loop
    elif kind == "rope":
        matching = match_rope
        resolving = resolve_rope
    else:
        raise ValueError()

    def resolver(current: Board) -> Resolving:
        links = set(search_hard(current))

        for chain in searching(links, matching):
            res = tuple(resolving(current, chain))
            if not res:
                # continue to search if if didn't work
                continue

            yield from res
            break

    resolver.__name__ = f"chains[{strategy}][{kind}]"

    return resolver

## A puzzle


In [ ]:
from utils import parse

# totally normal level puzzle:
# solvable by: basics + multiples[2, 3, 4] + chains[rope]
puzzle = parse("""
""")
# best result:
# 210 (m2 m3 m4 m5 [B2][rope])


In [ ]:
puzzle = Board.transform(puzzle, fillempty)

In [ ]:
# puzzle = await solve_silent(puzzle, cleanup, singles)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    cleanup,
    singles,
    clean_mults_(2),
    unhide_mults_(2),
    clean_mults_(3),
    unhide_mults_(3),
    clean_mults_(4),
    unhide_mults_(4),
    clean_mults_(5),
    unhide_mults_(5),
    chains_("rope"),
    chains_("loop"),
    filtout={"cleanup", "singles"},
)

### GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from collections import Counter
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display
from pprint import pprint

from traitlets import HasTraits, Instance, Set, Unicode, observe, Enum, Bool, Dict, Union
from canvas import SudokuCanvas

In [ ]:
"""GUI meta-widget"""

debug_view = w.Output()


def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


LINK_STYLES = {
    Link: "SOLID",
    HLink: "HARD",
    SLink: "SOFT",
}


class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Enum(["INCOMPLETE", "SOLVED", "BROKEN"])
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Target))
    finals = Set(Instance(Target))
    anchors = Set(Union((Instance(Target), Instance(MultiTarget))))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()
        self._counters = {
            dig: w.Label(
                "...",
                layout=dict(width="auto"),
            )
            for dig in DIGITS
        }
        self._counter_total = w.Label(
            "...",
            layout=dict(width="auto"),
        )
        self._status = w.Label(layout=dict(width="auto"), style=dict(text_color="white"))
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._counter_total, self._status],
                    layout=dict(align_items="stretch", width="6em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self.status = validate(self.puzzle)
        self.counters = count_finals(self.puzzle)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "transparent"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            self._counters[dig].value = f"({dig}): {cnt}"
            self._counters[dig].style.background = "var(--jp-success-color0)" if cnt == 9 else ""
        total = counters.total()
        self._counter_total.value = f"Total: {total}"
        self._counter_total.style.background = "var(--jp-success-color0)" if total == 81 else ""

    @observe("targets", "finals", "anchors", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for trg in self.targets:
                if isinstance(trg, Target):
                    self._canvas.highlight_segment(trg.loc, trg.dig, color="orange")

            for trg in self.finals:
                self._canvas.highlight_segment(trg.loc, trg.dig, color="purple")

            for lnk in self.links:
                t1, t2 = lnk
                self._canvas.highlight_link(t1.loc, t1.dig, t2.loc, t2.dig, style=LINK_STYLES[lnk.__class__], color="blue")

            for trg in self.anchors:
                if isinstance(trg, Target):
                    self._canvas.highlight_segment(trg.loc, trg.dig, color="cyan")
                if isinstance(trg, MultiTarget):
                    for loc in trg.loc.iter():
                        for dig in trg.digs:
                            self._canvas.highlight_segment(loc, dig, color="cyan")

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        with debug_view:
            self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False

In [ ]:
gui = GUI()

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver):
    result = initial
    gui.puzzle = result
    gui.running = True
    gui.inspecting = {r.__name__: True for r in resolvers}

    try:
        iteration = 0
        async for resolver, resolution, result in solver(initial, orchestrator(initial, *resolvers)):
            iteration += 1
            print(iteration, resolver.__name__)
            # pprint(resolution)
            if gui.inspecting[resolver.__name__]:
                resolving = f"#{iteration} {resolver.__name__}: "
                if resolution.castaways:
                    resolving += f"-= {len(resolution.castaways)}"
                if resolution.finals:
                    resolving += f"== {len(resolution.finals)}"
                gui.resolving = resolving
                render_resolution(resolution)
                await gui.pause()
                clear_resolution()

                gui.puzzle = result
                await asyncio.sleep(0.2)
            else:
                gui.puzzle = result
            gui.resolving = f"#{iteration}"
    except Exception as e:
        with debug_view:
            raise RuntimeError("Solver failed") from e

    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    gui.targets = res.castaways if res.castaways else set()
    gui.finals = res.finals if res.finals else set()
    if res.highlights is not None:
        gui.anchors = res.highlights.get("anchors", set())
        if "chain" in res.highlights:
            gui.links = set(res.highlights["chain"])
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def clear_resolution():
    gui.targets = set()
    gui.finals = set()
    gui.anchors = set()
    gui.links = set()


In [ ]:
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle

In [ ]:
puzzle = gui.puzzle

In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        cleanup,
        singles,
        clean_mults_(2),
        unhide_mults_(2),
        clean_mults_(3),
        unhide_mults_(3),
        clean_mults_(4),
        unhide_mults_(4),
        clean_mults_(5),
        unhide_mults_(5),
        chains_("rope"),
        chains_("loop"),
    )
)

In [ ]:
task

In [ ]:
task.cancel()

In [ ]:
puzzle = task.result()